# 🎮 DQN Atari Pong — Group 10 Experiments

**How to use this notebook:**
1. Go to **Runtime → Change runtime type → T4 GPU** (free!)
2. Run **Cell 1** once to install everything
3. Run **Cell 2** once to connect Google Drive
4. For each experiment, go to **Cell 3**, change the hyperparameters, then run it
5. Copy the printed results into your README table

---
**Member name guide:**
- Mitali → use experiments M1–M10
- Elissa → use experiments E1–E10  
- Caline → use experiments C1–C10

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# Run this ONCE at the start. Takes about 2 minutes.
# ============================================================
!pip install -q stable-baselines3[extra] gymnasium[atari,accept-rom-license] ale-py shimmy
!pip install -q autorom
!AutoROM --accept-license
print('\n✅ All dependencies installed!')

In [ ]:
# ============================================================
# CELL 2 — Connect Google Drive
# Run this ONCE. Your logs and models will be saved to Drive.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DQN_Group10'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)

print(f'✅ Google Drive connected!')
print(f'📁 Your files will be saved to: {DRIVE_DIR}')

In [ ]:
# ============================================================
# CELL 3 — RUN ONE EXPERIMENT
# Change the values below for each experiment, then run.
# ============================================================

# -----------------------------------------------------------
# 👤 FILL IN YOUR NAME AND EXPERIMENT ID
# -----------------------------------------------------------
MEMBER_NAME    = "Mitali"   # Change to your name
EXPERIMENT_ID  = "M1"       # Change to your experiment ID e.g. M1, M2, E1, C5 etc.

# -----------------------------------------------------------
# 🔧 HYPERPARAMETERS — Change these for each experiment
# -----------------------------------------------------------
POLICY               = "CnnPolicy"  # CnnPolicy or MlpPolicy
lr                   = 1e-4         # Learning rate
gamma                = 0.99         # Discount factor
batch_size           = 32           # Batch size
exploration_fraction = 0.1          # Fraction of training spent exploring
exploration_final_eps = 0.02        # Minimum epsilon (epsilon_end)
exploration_initial_eps = 1.0       # Starting epsilon (always 1.0)

# -----------------------------------------------------------
# ⚙️ TRAINING SETTINGS — Do not change these
# -----------------------------------------------------------
ENV_ID         = "ALE/Pong-v5"
TOTAL_TIMESTEPS = 50_000    # 50k steps = ~5-10 min on Colab GPU
SEED           = 42

# -----------------------------------------------------------
# 🚀 TRAINING — Do not change anything below this line
# -----------------------------------------------------------
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
import json, os

experiment_label = f"{MEMBER_NAME}_{EXPERIMENT_ID}"
LOG_DIR   = f"/content/drive/MyDrive/DQN_Group10/logs/{experiment_label}"
MODEL_DIR = f"/content/drive/MyDrive/DQN_Group10/models"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Save config
config = {
    "member": MEMBER_NAME,
    "experiment_id": EXPERIMENT_ID,
    "policy": POLICY,
    "lr": lr,
    "gamma": gamma,
    "batch_size": batch_size,
    "exploration_fraction": exploration_fraction,
    "exploration_final_eps": exploration_final_eps,
    "exploration_initial_eps": exploration_initial_eps,
    "total_timesteps": TOTAL_TIMESTEPS
}
with open(f"{LOG_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

class EpisodeStatsCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is not None:
                self.episode_rewards.append(float(ep["r"]))
                self.episode_lengths.append(int(ep["l"]))
                if len(self.episode_rewards) % 5 == 0:
                    avg_r = sum(self.episode_rewards[-5:]) / 5
                    avg_l = sum(self.episode_lengths[-5:]) / 5
                    print(f"  Episodes: {len(self.episode_rewards)} | "
                          f"Avg Reward (last 5): {avg_r:.2f} | "
                          f"Avg Length (last 5): {avg_l:.0f}")
        return True

def make_env(seed):
    env = make_atari_env(ENV_ID, n_envs=1, seed=seed)
    return VecFrameStack(env, n_stack=4)

print(f"\n{'='*60}")
print(f"🧪 EXPERIMENT {EXPERIMENT_ID} — {MEMBER_NAME}")
print(f"{'='*60}")
print(f"  Policy:               {POLICY}")
print(f"  Learning Rate:        {lr}")
print(f"  Gamma:                {gamma}")
print(f"  Batch Size:           {batch_size}")
print(f"  Epsilon Start:        {exploration_initial_eps}")
print(f"  Epsilon End:          {exploration_final_eps}")
print(f"  Exploration Fraction: {exploration_fraction}")
print(f"  Total Timesteps:      {TOTAL_TIMESTEPS}")
print(f"{'='*60}\n")

train_env = make_env(SEED)
eval_env  = make_env(SEED + 100)

model = DQN(
    policy=POLICY,
    env=train_env,
    learning_rate=lr,
    gamma=gamma,
    batch_size=batch_size,
    exploration_fraction=exploration_fraction,
    exploration_final_eps=exploration_final_eps,
    exploration_initial_eps=exploration_initial_eps,
    tensorboard_log=LOG_DIR,
    seed=SEED,
    verbose=0,
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=LOG_DIR,
    eval_freq=10_000,
    deterministic=True,
    render=False,
    n_eval_episodes=5,
)

stats_cb = EpisodeStatsCallback()

print("⏳ Training started...\n")
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_callback, stats_cb],
    tb_log_name=experiment_label,
)

model.save(f"{MODEL_DIR}/{experiment_label}_model")

# Final summary
all_rewards = stats_cb.episode_rewards
if all_rewards:
    mean_reward = sum(all_rewards) / len(all_rewards)
    last5_reward = sum(all_rewards[-5:]) / min(5, len(all_rewards))
    min_reward = min(all_rewards)
    max_reward = max(all_rewards)
else:
    mean_reward = last5_reward = min_reward = max_reward = 0

print(f"\n{'='*60}")
print(f"✅ EXPERIMENT {EXPERIMENT_ID} COMPLETE!")
print(f"{'='*60}")
print(f"  Total Episodes:       {len(all_rewards)}")
print(f"  Mean Reward:          {mean_reward:.2f}")
print(f"  Last 5 Avg Reward:    {last5_reward:.2f}")
print(f"  Min Reward:           {min_reward:.2f}")
print(f"  Max Reward:           {max_reward:.2f}")
print(f"{'='*60}")
print(f"\n📋 COPY THIS INTO YOUR README TABLE:")
print(f"| {MEMBER_NAME} | {EXPERIMENT_ID} | {POLICY} | {lr} | {gamma} | "
      f"{batch_size} | {exploration_initial_eps} | {exploration_final_eps} | "
      f"{exploration_fraction} | {last5_reward:.2f} | <your observation here> |")
print(f"\n💾 Model saved to Google Drive: {MODEL_DIR}/{experiment_label}_model.zip")
print(f"📁 Logs saved to Google Drive: {LOG_DIR}")

train_env.close()
eval_env.close()

---
## 📋 Your 10 Experiment Configs

Copy these values into Cell 3 one at a time.

### 👤 Mitali (M1–M10)
| ID | POLICY | lr | gamma | batch_size | exploration_fraction | exploration_final_eps |
|----|--------|----|-------|------------|----------------------|-----------------------|
| M1 | CnnPolicy | 1e-4 | 0.99 | 32 | 0.10 | 0.02 |
| M2 | CnnPolicy | 1e-3 | 0.99 | 32 | 0.10 | 0.02 |
| M3 | CnnPolicy | 1e-5 | 0.99 | 32 | 0.10 | 0.02 |
| M4 | CnnPolicy | 1e-4 | 0.95 | 32 | 0.10 | 0.02 |
| M5 | CnnPolicy | 1e-4 | 0.999 | 32 | 0.10 | 0.02 |
| M6 | CnnPolicy | 1e-4 | 0.99 | 64 | 0.10 | 0.02 |
| M7 | CnnPolicy | 1e-4 | 0.99 | 16 | 0.10 | 0.02 |
| M8 | CnnPolicy | 1e-4 | 0.99 | 32 | 0.20 | 0.02 |
| M9 | CnnPolicy | 1e-4 | 0.99 | 32 | 0.10 | 0.10 |
| M10 | MlpPolicy | 1e-4 | 0.99 | 32 | 0.10 | 0.02 |

### 👤 Elissa (E1–E10)
| ID | POLICY | lr | gamma | batch_size | exploration_fraction | exploration_final_eps |
|----|--------|----|-------|------------|----------------------|-----------------------|
| E1 | CnnPolicy | 5e-4 | 0.99 | 32 | 0.10 | 0.05 |
| E2 | CnnPolicy | 5e-4 | 0.98 | 32 | 0.10 | 0.05 |
| E3 | CnnPolicy | 5e-4 | 0.99 | 128 | 0.10 | 0.05 |
| E4 | CnnPolicy | 5e-4 | 0.99 | 32 | 0.15 | 0.05 |
| E5 | CnnPolicy | 2e-4 | 0.99 | 32 | 0.10 | 0.05 |
| E6 | CnnPolicy | 2e-4 | 0.95 | 64 | 0.10 | 0.05 |
| E7 | CnnPolicy | 2e-4 | 0.99 | 32 | 0.25 | 0.05 |
| E8 | CnnPolicy | 1e-3 | 0.98 | 64 | 0.10 | 0.05 |
| E9 | CnnPolicy | 1e-4 | 0.99 | 32 | 0.10 | 0.20 |
| E10 | CnnPolicy | 5e-4 | 0.999 | 32 | 0.05 | 0.01 |

### 👤 Caline (C1–C10)
| ID | POLICY | lr | gamma | batch_size | exploration_fraction | exploration_final_eps |
|----|--------|----|-------|------------|----------------------|-----------------------|
| C1 | CnnPolicy | 3e-4 | 0.99 | 32 | 0.10 | 0.01 |
| C2 | CnnPolicy | 3e-4 | 0.90 | 32 | 0.10 | 0.01 |
| C3 | CnnPolicy | 3e-4 | 0.99 | 256 | 0.10 | 0.01 |
| C4 | CnnPolicy | 3e-4 | 0.99 | 32 | 0.30 | 0.01 |
| C5 | CnnPolicy | 3e-4 | 0.99 | 32 | 0.10 | 0.50 |
| C6 | CnnPolicy | 1e-3 | 0.99 | 128 | 0.20 | 0.01 |
| C7 | CnnPolicy | 1e-5 | 0.99 | 32 | 0.10 | 0.01 |
| C8 | CnnPolicy | 3e-4 | 0.99 | 32 | 0.10 | 0.001 |
| C9 | CnnPolicy | 3e-4 | 0.97 | 64 | 0.15 | 0.05 |
| C10 | MlpPolicy | 3e-4 | 0.99 | 32 | 0.10 | 0.01 |


---
## 💡 What To Write In 'Noted Behavior'

After each experiment look at the **Last 5 Avg Reward** printed at the end.

In Pong, rewards range from **-21 (worst) to +21 (best)**:

| Last 5 Avg Reward | What It Means | What To Write |
|-------------------|---------------|---------------|
| Around -21 | Agent learned nothing | Agent not learning, always losing |
| Around -15 to -20 | Very early learning | Slow learning, mostly losing |
| Around -10 to -14 | Some learning | Agent improving, losing less |
| Around -5 to -9 | Good learning | Agent learning well |
| Around 0 to +5 | Strong learning | Agent competitive |
| Above +5 | Excellent | Agent winning most points |